In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
# RQ5-KN01: Docling parser variants (RQ5a) on the 5 digital and 10 scanned sample PDFs, chunks with 5 chunkers, vectors
!git clone -q https://github.com/amanchaurasia512/Self_optimized_Agent_Ai_chunking.git /kaggle/working/repo
!cd /kaggle/working/repo && git log --oneline -1
!apt-get -qq update && apt-get -qq install -y tesseract-ocr tesseract-ocr-eng tesseract-ocr-fra tesseract-ocr-deu tesseract-ocr-spa > /dev/null && tesseract --version | head -1
!pip install -q docling==2.143.0 jsonschema langchain-text-splitters==1.1.3 langchain-experimental==0.4.2 langchain-community==0.4.2 sentence-transformers==6.2.0 transformers==5.22.1 stanza==1.15.0 google-genai==2.34.0 rapidocr onnxruntime
!pip list 2>/dev/null | grep -i "^docling \|rapidocr\|onnxruntime"
import torch
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
import os
import shutil
import sys
import zipfile
from pathlib import Path

import pandas as pd
from kaggle_secrets import UserSecretsClient

REPO, INPUT, TEMP = Path("/kaggle/working/repo"), Path("/kaggle/input"), Path("/kaggle/temp")
sys.path.insert(0, str(REPO))
for archive in INPUT.rglob("*.zip"):
    zipfile.ZipFile(archive).extractall(TEMP / archive.stem)
roots = [INPUT, TEMP]
DIGITAL = list(pd.read_csv(REPO / "pdfs" / "rq5_digital_picks.csv")["doc_id"])
split = pd.read_csv(REPO / "pdfs" / "split.csv")
SCANNED = list(split.loc[split["split"] == "scanned_rq5", "doc_id"])

PDFS = Path("/kaggle/working/rq5_pdfs")
PDFS.mkdir(exist_ok=True)
for path in (path for root in roots for path in root.rglob("*.pdf") if path.stem in DIGITAL + SCANNED):
    shutil.copyfile(path, PDFS / path.name)
for parser in ["pdfplumber_layout", "pdfplumber_tolerance_wide"]:
    target = REPO / "data" / "extracted" / parser
    target.mkdir(parents=True, exist_ok=True)
    for path in (path for root in roots for path in root.rglob(f"{parser}/*.json")):
        shutil.copyfile(path, target / path.name)
    print(parser, ":", len(list(target.glob("*.json"))), "of", len(DIGITAL))
FROZEN = next(path for root in roots for path in root.rglob("frozen_docling/*.json"))
print("PDFs:", len(list(PDFS.glob("*.pdf"))), "of", len(DIGITAL + SCANNED), "| identity doc:", FROZEN.stem)

GOLD = next(path for root in roots for path in root.rglob("gold") if list(path.glob("spans_*.jsonl")))
CACHE = Path("/kaggle/working/embeddings/bge-m3")
source = next(Path(folder) for root in roots for folder, subfolders, files in os.walk(root, followlinks=True)
              if "embedder.json" in files and "embeddings_test" not in folder)
shutil.rmtree(CACHE, ignore_errors=True)
shutil.copytree(source, CACHE)
print("gold:", GOLD, "| cache:", len(list(CACHE.glob("*.npy"))), "vector part(s)")

key_file = Path.home() / ".config" / "gemini" / "api_key"
key_file.parent.mkdir(parents=True, exist_ok=True)
key_file.write_text(UserSecretsClient().get_secret("GEMINI_API_KEY").strip())
key_file.chmod(0o600)

In [ ]:
import json
import subprocess

check = Path("/kaggle/working/identity_check")
result = subprocess.run(f"python -m scripts.p1_extract_docling --input {PDFS} --output {check} --only {FROZEN.stem} "
                        f"--variant default --machine kaggle --run-source RQ5-KN01-check", shell=True, cwd=REPO, capture_output=True, text=True)
print("exit", result.returncode, (result.stdout + result.stderr).strip().splitlines()[-1:])
new, frozen = (json.loads(path.read_text(encoding="utf-8")) for path in [check / f"{FROZEN.stem}.json", FROZEN])
same = {key: new[key] == frozen[key] for key in ["text", "pages", "elements", "parser"]}
print("default identical to frozen:", same)
assert all(same.values()), "Docling default does not reproduce the frozen extraction: stopping"

In [ ]:
DIGITAL_VARIANTS = ["ocr_off", "table_mode_switched", "headings", "replace_broken_text"]
SCANNED_VARIANTS = ["default", "ocr_tesseract", "ocr_rapidocr", "ocr_scale_double"]
PLAN = [(variant, DIGITAL) for variant in DIGITAL_VARIANTS] + [(variant, SCANNED) for variant in SCANNED_VARIANTS]
failed = []
for variant, docs in PLAN:
    output = REPO / "data" / "extracted" / ("docling" if variant == "default" else f"docling_{variant}")
    command = (f"python -m scripts.p1_extract_docling --input {PDFS} --output {output} --only {' '.join(docs)} "
               f"--variant {variant} --machine kaggle --run-source RQ5-KN01")
    result = subprocess.run(command, shell=True, cwd=REPO, capture_output=True, text=True)
    lines = (result.stdout + result.stderr).strip().splitlines()
    done = len([path for path in output.glob("*.json") if path.stem in docs])
    print(f"=== {variant} | exit {result.returncode} | {done} of {len(docs)} extracted\n" + "\n".join(lines[-3:]) + "\n")
    if result.returncode or done != len(docs):
        failed.append(variant)
print("failed:", failed or "none")

In [ ]:
SCANNED_PARSERS = {"docling": SCANNED, **{f"docling_{variant}": SCANNED for variant in SCANNED_VARIANTS[1:]}}
PARSERS = {**{f"docling_{variant}": DIGITAL for variant in DIGITAL_VARIANTS},
           "pdfplumber_layout": DIGITAL, "pdfplumber_tolerance_wide": DIGITAL, **SCANNED_PARSERS}
CHUNKERS = [("method1_tree", "T=256"), ("method1_tree", "T=384"), ("recursive", "size=256 overlap=0"),
            ("langchain_recursive", "size=256 overlap=0"), ("fixed_token", "size=256 overlap=0")]
skipped, chunk_failed = [], []
for parser, docs in PARSERS.items():
    folder = REPO / "data" / "extracted" / parser
    if len([path for path in folder.glob("*.json") if path.stem in docs]) != len(docs):
        skipped.append(parser)
        continue
    for chunker, settings in CHUNKERS:
        command = (f"python -m scripts.p2_chunk --chunker {chunker} --set {settings} --parsers {parser} --docs {' '.join(docs)} "
                   f"--final --machine kaggle --run-source RQ5-KN01")
        result = subprocess.run(command, shell=True, cwd=REPO, capture_output=True, text=True)
        print(f"=== {parser} | {chunker} {settings} | exit {result.returncode}")
        if result.returncode:
            chunk_failed.append(f"{parser} {chunker} {settings}")
            print("\n".join((result.stdout + result.stderr).strip().splitlines()[-3:]))
print("skipped (extraction missing):", skipped or "none", "| failed:", chunk_failed or "none")

In [ ]:
!cd /kaggle/working/repo && python -m scripts.p3_embed --chunks /kaggle/working/repo/data/chunks --gold {GOLD} --cache {CACHE} --name RQ5-KN01 --run-source RQ5-KN01 --final